In [1]:
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

In [2]:
# Load dataset

df = pd.read_csv("../data/processed/reviews_cleaned.csv")
df = df.drop_duplicates(subset=["combined_text"])

print(len(df))
df.head()

6388


,review_text,label,rating,source,combined_text
0,earn money fast and win exciting prizes!!!!!!,1,3,clean,earn money fast and win exciting prizes!!!!!!....
1,visit this link to get this product now!!!,1,3,clean,visit this link to get this product now!!!
2,"This laptop is okay, it works well.",0,4,clean,"This laptop is okay, it works well."
3,"THIS LAPTOP IS OKAY, IT STOPPED WORKING. THIS ...",0,4,clean,"THIS LAPTOP IS OKAY, IT STOPPED WORKING. THIS ..."
4,I bought this camera recently and it stopped w...,0,2,clean,I bought this camera recently and it stopped w...


In [3]:
# Train test split

# Split clean vs noisy

df_clean = df[df["source"] == "clean"]
df_noisy = df[df["source"] == "noisy"]

# Validation ONLY from clean data (no duplicates)
val_df = df_clean.sample(frac=0.1, random_state=42)

# Remove validation from clean training
train_clean = df_clean.drop(val_df.index)

# Training = remaining clean + noisy
train_df = pd.concat([train_clean, df_noisy])

train_texts = train_df["combined_text"]
train_labels = train_df["label"]

val_texts = val_df["combined_text"]
val_labels = val_df["label"]

print("Train size:", len(train_df))
print("Validation size (clean only):", len(val_df))

Train size: 5996
Validation size (clean only): 392


In [4]:
# Tokenization

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding="longest",
    max_length=64
)

val_encodings = tokenizer(
    val_texts.tolist(),
    truncation=True,
    padding="longest",
    max_length=64
)
# Debug labels

print(train_labels.head())
print(type(train_labels.iloc[0]))

0    1
1    1
2    0
3    0
4    0
Name: label, dtype: int64
<class 'numpy.int64'>


In [5]:
# Fix label mapping

def map_label(x):
    x = str(x).lower()
    if x == "fake":
        return 1
    elif x == "genuine":
        return 0
    else:
        return int(x)

train_labels = train_labels.apply(map_label)
val_labels = val_labels.apply(map_label)

print(train_labels.unique())

[1 0]


In [6]:
# Dataset class

class ReviewDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]))
        return item

    def __len__(self):
        return len(self.labels)
    

In [7]:
# DataLoader

train_dataset = ReviewDataset(train_encodings, train_labels)
val_dataset = ReviewDataset(val_encodings, val_labels)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    num_workers=0,
    pin_memory=False
)

In [8]:
# Load model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [9]:
# Optimizer

optimizer = AdamW(model.parameters(), lr=5e-5)

In [10]:
# Training loop

model.train()

for epoch in range(1):
    total_loss = 0

    for i, batch in enumerate(train_loader):

        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if i % 200 == 0:
            print(f"Epoch {epoch+1} Batch {i} Loss {loss.item():.4f}")

    print(f"Epoch {epoch+1} Total Loss {total_loss:.4f}")

Epoch 1 Batch 0 Loss 0.6952
Epoch 1 Batch 200 Loss 0.1997
Epoch 1 Total Loss 64.9525


In [11]:
# Save model

model.save_pretrained("../models/distilbert_v2")
tokenizer.save_pretrained("../models/distilbert_v2")

print("Model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved
